# Addition Interp

by ravi

### Setup / Model Verification

In [108]:
import torch as t
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import einops

import typing
from typing import Literal, List

from dataclasses import dataclass

import numpy as np

import plotly.express as px

from utils import (
    vocab,
    vocab_inv,
    pad,
    generate_sample,
    dataConfig,
    data_cfg,
    samples_tensor, 
    SumDataset,
    ds,
    train_ds,
    val_ds,
    train_dl,
    val_dl,
    create_model,
)

DEVICE = 'cpu'

In [51]:
model = create_model(
    num_digits=4,
    seed=0,
    d_model=48,
    d_head=24,
    n_layers=2,
    n_heads=3,
    normalization_type="LN",
    d_mlp=None,
    device=DEVICE,
)

In [52]:
model.load_state_dict(t.load('../models/sum_model.pt', map_location=DEVICE, weights_only=False))
model.eval()
print('Model loaded.')

Model loaded.


In [90]:
sample = samples_tensor[2].unsqueeze(0)
logits = model(sample).detach()

In [91]:
ANS = slice(9, 13)

def masked_loss(logits, tokens):
    logp = logits[:, ANS].log_softmax(-1)
    tgt  = tokens[:, 10:14]
    return -logp.gather(-1, tgt.unsqueeze(-1)).mean()

In [92]:
digit_probs = logits.squeeze()[ANS].softmax(-1)

px.imshow(
    digit_probs,
    title=f'Digit probabilities for solution : {sample.squeeze()[10:14].tolist()} with observed loss : {masked_loss(logits, sample):.5f}',
    labels=dict(x='Digit', y='Digit Position'),
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],)

### 1. Inspecting Attention Heads

In [113]:
LOGIT_IDX = slice(9, 13)
SUM_IDX = slice(10, 14)

In [ ]:
# Run the model over one hundred examples.

samples = ds[0:100]['tokens']
logits, cache = model.run_with_cache(samples, return_cache_object=True)

In [125]:
attn_scores_0 = cache['blocks.0.attn.hook_pattern']
attn_scores_1= cache['blocks.1.attn.hook_pattern']

avg_scores_0 = einops.reduce(attn_scores_0, 'b h seq_Q seq_K -> h seq_Q seq_K', reduction='mean')
avg_scores_1 = einops.reduce(attn_scores_1, 'b h seq_Q seq_K -> h seq_Q seq_K', reduction='mean')

avg_scores = t.stack([avg_scores_0, avg_scores_1], dim=0)
sum_avg_scores = avg_scores[:, :, LOGIT_IDX, :]

In [126]:
px.imshow(
    sum_avg_scores.detach().cpu().numpy(),
    facet_col=0,
    facet_row=1,
    labels=dict(x='Key', y='Query'),
    zmin=0,
    zmax=1,
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],
)